# Lab12 프로그래밍 과제

# Multi-Layer Perceptron with MNIST handwritten digits classification

# Forward Propagation 구현하기

## 1. Module Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision import datasets
import torchvision.transforms as transforms
import torch.nn.functional as F

## 2. 딥러닝 모델을 설계할 때 활용하는 장비 확인

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Using PyTorch version:', torch.__version__, ' Device:', device)

Using PyTorch version: 2.3.0+cu121  Device: cpu


## 3. MNIST 데이터 다운로드 (Train data와 Test data 분리하기)

In [ ]:
BATCH_SIZE = 32

train_data = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset=test_data, batch_size=BATCH_SIZE, shuffle=False)


Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9912422/9912422 [00:00<00:00, 53660411.55it/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28881/28881 [00:00<00:00, 1832806.71it/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1648877/1648877 [00:00<00:00, 13874516.30it/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4542/4542 [00:00<00:00, 2739112.69it/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



## 4. 첫번째 batch 데이터의 크기와 타입을 확인하기

In [ ]:
for (X_train, y_train) in train_loader:
    print('X_train:', X_train.size(), 'type:', X_train.type())
    print('y_train:', y_train.size(), 'type:', y_train.type())
    break

X_train: torch.Size([32, 1, 28, 28]) type: torch.FloatTensor
y_train: torch.Size([32]) type: torch.LongTensor


# Loss 구하기
   forward propagation을 계산하는 함수 구현하기
   
   1) input layer (입력층), hidden layer (은닉층), output layer (출력층) 으로 이루어진 모델을 이용

   2) 하나의 hidden layer (은닉층)만 이용 - 은닉층의 개수는 100개로 하세요

   3) 모든 것은 tensor 계산으로만 할 것!!


## ReLU, one_hot_encoding, softmax, cross_entropy 구하기

아래 코드는 각 함수가 맞는지 확인하기 위해서 만든 임의의 값입니다.
각 함수가 작동을 잘하는지 확인해 보세요.

In [ ]:
test_data = torch.tensor([[1,-2,-4, 2, 5, 6, -3, -5, 0, 2],[2, -3, 4, 3, -1, -4, 3, 5, 2, -3]])
true_label = torch.tensor([5,7])
false_label = torch.tensor([1,0])

# Problem 1 (1 points)

ReLU 함수를 구현하세요.

In [ ]:
def ReLU_func(outputs):
    zero_tensor = torch.zeros(2,10)
    final_outputs =torch.maximum(zero_tensor,outputs) #여기를 채우세요 torch.maximum 함수를 사용하세요

    return final_outputs

ReLU 함수가 맞는지 test_data를 이용하여 맞추어 보자. 아래 함수의 결과는 어떻게 예상되는가?

In [ ]:
ReLU_func(test_data)

tensor([[1., 0., 0., 2., 5., 6., 0., 0., 0., 2.],
        [2., 0., 4., 3., 0., 0., 3., 5., 2., 0.]])

# Problem 2 (2 points)

one-hot encoding 함수를 구현하세요.

In [ ]:
def one_hot_encoding(label):

    one_hot_outputs =torch.eye(10)[label] #여기를 채우세요 torch.eye를 사용하세요

    return one_hot_outputs

one_hot_encoding 함수가 맞는지 true_label, false_label을 이용하여 맞추어 보자. 아래 함수 결과는 어떻게 예상되는가?

In [ ]:
# 검증을 위해 아래 4줄을 사용하면 됩니다.
tl = one_hot_encoding(true_label)
print(tl)
fl = one_hot_encoding(false_label)
print(fl)

# 나중을 위해서 사용될 코드 입니다.
close_tl = tl.clone()
close_tl[[0,1],true_label] -= 0.001
close_tl[false_label] += 0.0001
print(close_tl)

tensor([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])
tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 9.9910e-01,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 9.9910e-01, 1.0000e-04, 1.0000e-04]])


# Problem 3 (2 points)

softmax 함수를 구현하세요.

In [ ]:
def softmax(outputs):
    numerator = torch.exp(outputs-torch.max(outputs,dim=1,keepdim=True)[0])#여기를 채우세요, 1) 2차원의 output도 계산할수 있게 2) 또한 overflow를 신경쓰면서)
    denominator = torch.sum(numerator,dim=1,keepdim=True)#여기를 채우세요).view(-1,1)
    softmax = numerator/denominator

    return softmax

softmax 함수가 맞는지 test_data를 이용하여 맞추어 보자. 아래 함수의 결과는 어떻게 예상되는가?

In [ ]:
a = softmax(test_data)
print(a)
torch.sum(a,axis=1)

tensor([[4.7643e-03, 2.3720e-04, 3.2102e-05, 1.2951e-02, 2.6012e-01, 7.0709e-01,
         8.7262e-05, 1.1810e-05, 1.7527e-03, 1.2951e-02],
        [2.8590e-02, 1.9264e-04, 2.1126e-01, 7.7716e-02, 1.4234e-03, 7.0868e-05,
         7.7716e-02, 5.7425e-01, 2.8590e-02, 1.9264e-04]])


tensor([1., 1.])

# Problem 4 (2 points)

cross entropy 오차 함수를 구현하세요.

In [ ]:
def cross_entropy(outputs, labels):
  exps=torch.exp(outputs-torch.max(outputs,dim=1,keepdim=True).values)
  softmax_outputs = exps / torch.sum(exps, dim=1, keepdim=True)
  log_softmax_outputs = torch.log(softmax_outputs)
  cross_entropy_loss = -torch.sum(labels * log_softmax_outputs, dim=1)
  return torch.mean(cross_entropy_loss) #여기를 채우세요

cross_entropy 함수가 맞는지 test_data를 이용하여 맞추어 보자. 아래 함수의 결과는 어떻게 예상되는가? tl, fl, close_tl을 이용하여 각각의 cross entropy를 구하고 그 값이 맞는지 확인하세요

In [ ]:
ideal_result = cross_entropy(close_tl, tl)
non_ideal_result = cross_entropy(close_tl,fl)

print(ideal_result)
print(non_ideal_result)

true_result = cross_entropy(a,tl)
false_result = cross_entropy(a,fl)

print(true_result)
print(false_result)


tensor(1.4619)
tensor(2.4609)
tensor(1.7837)
tensor(2.4100)


# Problem 5 (3 points)

Forward propagation 과정을 구현하세요.

In [ ]:
import torch
import torch.nn.functional as F

def one_hot_encoding(label, num_classes=10):
    return torch.eye(num_classes)[label]

def forward_pass(train_loader):
    for batch_idx, (image, label) in enumerate(train_loader):
        # Process only the first image and label from the batch
        image = image[0].view(-1, 28 * 28)  # Flatten the image
        label = label[0]

        print("Image Size:", image.size())

        # Initialize weights for the input-to-hidden layer and hidden-to-output layer
        W_ih = torch.rand(28 * 28, 128) - 0.5  # Assuming 128 hidden units
        W_ho = torch.rand(128, 10) - 0.5  # Assuming 10 output classes

        # Forward propagation
        hidden_inputs = torch.matmul(image, W_ih)  # Input to the hidden layer
        hidden_outputs = F.relu(hidden_inputs)  # Activation function (ReLU)
        print("Hidden Layer Output Size:", hidden_outputs.size())

        # Output layer calculation
        final_inputs = torch.matmul(hidden_outputs, W_ho)  # Input to the output layer
        softmaxed_outputs = F.softmax(final_inputs, dim=0)  # Softmax to get probabilities
        print("Softmax Output Size:", softmaxed_outputs.size())

        # Convert label to one-hot encoded format
        expected_outputs = one_hot_encoding(label)

        # Calculate the loss
        loss = torch.mean(-torch.sum(expected_outputs * torch.log(softmaxed_outputs), dim=0))  # Cross-entropy loss
        print("Loss:", loss)

        break  # Only process the first batch

# Usage example: you would need to define your train_loader properly with torchvision datasets and DataLoader
# train_loader = DataLoader(...)
# forward_pass(train_loader)


In [ ]:
forward_pass(train_loader) # 결과물이 맞다면 아래 Markdown cell과 같은 값이 나오게 됩니다. 단, 4번째 줄의 값은 변동 가능합니다.

Image Size: torch.Size([1, 784])
Hidden Layer Output Size: torch.Size([1, 128])
Softmax Output Size: torch.Size([1, 10])
Loss: tensor(0.)


torch.Size([1, 784])

torch.Size([1, 100])

torch.Size([1, 10])

tensor([13.0287])